# 010 Full TDGL Simulation Workflow

Configure -> Preview -> Submit -> Watch in real-time

- Device and timing use the same code as the K8s pods
- Device preview submitted through the `rectangle-device-builder` workflow (same as `device_builder.ipynb`)
- Timing preview shows the current sweep schedule
- After submit, watch the heatmap update in real-time as the solver runs

In [ ]:
import io
import json
import sys
import tarfile
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import boto3
import httpx
import numpy as np
import plotly.graph_objects as go

from hera.workflows import Workflow, WorkflowsService, Parameter
from hera.workflows.models import WorkflowTemplateRef as WTR

sys.path.insert(0, str(Path("src")))
from tdgl_sdk import TDGLRunStore
from tdgl_workflow.timing import build_timing

# Connections
gateway = "http://localhost:30080"
argo_svc = WorkflowsService(host=gateway, verify_ssl=False, namespace="tdgl")
minio = boto3.client(
    "s3",
    endpoint_url="http://localhost:30900",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
)
store = TDGLRunStore(
    endpoint_url="http://localhost:30900",
    access_key="minioadmin",
    secret_key="minioadmin123",
    bucket="tdgl-results",
)
print(f"Gateway: {gateway}")
print(f"MinIO:   http://localhost:30900")

## 1. Device Builder

Submit mesh generation through the `rectangle-device-builder` Argo workflow,
poll until done, then plot the mesh. Same flow as `device_builder.ipynb`.

In [ ]:
device_params = {
    "film_width": 10.0,
    "film_height": 4.0,
    "elec_width": 0.1,
    "elec_height": 4.1,
    "elec_y_offset": 0.0,
    "probe_points": [[-1.0, 0.0], [1.0, 0.0]],
    "max_edge_length": 0.25,
    "smooth": 100,
}

device_run_id = str(uuid.uuid4())

wf = Workflow(
    generate_name="rect-device-",
    namespace="tdgl",
    workflow_template_ref=WTR(name="rectangle-device-builder"),
    arguments=[
        Parameter(name="run-id", value=device_run_id),
        Parameter(name="device-params-json", value=json.dumps(device_params)),
    ],
    workflows_service=argo_svc,
)
created = wf.create()
device_wf_name = created.metadata.name
print(f"Submitted: {device_wf_name}")
print(f"Run ID:    {device_run_id}")

In [ ]:
hint_map = {
    "Submitted": "Scheduling...",
    "Pending": "Pulling image...",
    "Running": "Computing mesh...",
}

while True:
    url = f"{argo_svc.host}/api/v1/workflows/tdgl/{device_wf_name}"
    resp = httpx.get(url, verify=False, timeout=10)
    resp.raise_for_status()
    phase = (resp.json().get("status") or {}).get("phase", "Unknown")
    if phase == "Succeeded":
        print(f"{device_wf_name} succeeded.")
        break
    elif phase in {"Failed", "Error"}:
        raise RuntimeError(f"{device_wf_name} {phase}")
    else:
        hint = hint_map.get(phase, "Processing...")
        print(f"{device_wf_name} [{phase}] {hint}")
        time.sleep(3)

In [ ]:
# Read device.pkl artifact from MinIO (Argo wraps artifacts in tar.gz)
import pickle

key = f"{device_run_id}/device.pkl"
resp = minio.get_object(Bucket="argo-artifacts", Key=key)
raw = resp["Body"].read()

device_obj = None
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    for member in tar.getmembers():
        f = tar.extractfile(member)
        if f:
            device_obj = pickle.loads(f.read())
            break

sites = np.asarray(device_obj.points)
elements = np.asarray(device_obj.triangles)
probes = []
for px, py in device_params["probe_points"]:
    distances = np.sqrt((sites[:, 0] - px) ** 2 + (sites[:, 1] - py) ** 2)
    probes.append(int(np.argmin(distances)))

terminal_info = device_obj.terminal_info()
terminals = []
for t in terminal_info:
    terminals.append({
        "name": t.name,
        "site_indices": t.site_indices.tolist(),
    })

em = device_obj.mesh.edge_mesh

print(f"Sites: {len(sites)}  Elements: {len(elements)}")
print(f"Film: {device_params['film_width']} x {device_params['film_height']}")
print(f"Probes: {probes}")

In [ ]:
# Plot mesh (same style as device_builder.ipynb)
mx, my = [], []
for tri in elements:
    for j in range(3):
        p0, p1 = sites[tri[j]], sites[tri[(j + 1) % 3]]
        mx += [p0[0], p1[0], None]
        my += [p0[1], p1[1], None]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mx, y=my, mode="lines",
    line=dict(width=0.3, color="#94a3b8"),
    hoverinfo="skip", showlegend=False,
))

ec = {
    "source": ("#2563eb", "rgba(37,99,235,0.35)"),
    "drain": ("#dc2626", "rgba(220,38,38,0.35)"),
}
for t in terminals:
    idx = t["site_indices"]
    x0, x1 = sites[idx, 0].min(), sites[idx, 0].max()
    y0, y1 = sites[idx, 1].min(), sites[idx, 1].max()
    pad = 0.15
    lc, fc = ec.get(t["name"], ("#888", "rgba(136,136,136,0.35)"))
    fig.add_trace(go.Scatter(
        x=[x0 - pad, x1 + pad, x1 + pad, x0 - pad, x0 - pad],
        y=[y0 - pad, y0 - pad, y1 + pad, y1 + pad, y0 - pad],
        mode="lines", line=dict(width=1.5, color=lc), name=t["name"],
        fill="toself", fillcolor=fc,
    ))

fig.add_trace(go.Scatter(
    x=sites[probes, 0], y=sites[probes, 1],
    mode="markers+text",
    marker=dict(size=8, symbol="x", color="#16a34a", line_width=2),
    text=[f"P{i+1}" for i in range(len(probes))],
    textposition="top center", name="probes",
))

xmin, xmax = sites[:, 0].min(), sites[:, 0].max()
ymin, ymax = sites[:, 1].min(), sites[:, 1].max()
m = 0.3
fig.update_layout(
    title=f"Device ({len(sites)} sites, {len(elements)} elements)",
    xaxis=dict(range=[xmin - m, xmax + m], showline=True, linewidth=1,
               linecolor="black", mirror=True, ticks="outside"),
    yaxis=dict(scaleanchor="x", scaleratio=1, range=[ymin - m, ymax + m],
               showline=True, linewidth=1, linecolor="black", mirror=True, ticks="outside"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    margin=dict(l=40, r=10, t=35, b=50),
    height=280, width=700, plot_bgcolor="white",
)
fig.show()

## 2. Timing

Same `build_timing()` used by the `py-tdgl-runner` pod.

In [ ]:
timing_params = {
    "je_initial": 0.0,
    "je_final": 11.0,
    "je_step": 0.2,
    "ramp_time": 100.0,
    "stable_time": 100.0,
    "save_time": 20.0,
    "ramp_down": True,
}

timing_data = build_timing(**timing_params)
print(f"Timing: {timing_data['n_steps']} steps, solve_time={timing_data['solve_time']:.2f}s")

In [ ]:
steps = timing_data["steps"] + timing_data.get("ramp_down_steps", [])

fig = go.Figure()
times, jes = [], []
for s in steps:
    times.extend([s["ramp_start"], s["ramp_end"], s["stable_end"]])
    jes.extend([s["je_start"], s["je_end"], s["je_end"]])

fig.add_trace(go.Scatter(
    x=times, y=jes, mode="lines", name="Je", line=dict(width=2),
))

for s in steps:
    fig.add_vrect(
        x0=s["save_start"], x1=s["save_end"],
        fillcolor="green", opacity=0.1, line_width=0,
    )

fig.update_layout(
    title=f"Timing: {len(steps)} steps, solve_time={timing_data['solve_time']:.1f}s (green = save windows)",
    xaxis_title="Time (s)", yaxis_title="Je",
    width=800, height=350,
)
fig.show()

## 3. Submit Simulation Workflow

In [ ]:
solver_options = {
    "dt": 1e-6,
    "max_dt": 0.1,
    "save_every": 100,
}

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]

sim_wf = Workflow(
    generate_name=f"py-tdgl-sim-{run_id[:13]}-",
    namespace="tdgl",
    workflow_template_ref=WTR(name="py-tdgl-sim"),
    arguments=[
        Parameter(name="run-id", value=run_id),
        Parameter(name="device-params-json", value=json.dumps(device_params)),
        Parameter(name="timing-params-json", value=json.dumps(timing_params)),
        Parameter(name="solver-options-json", value=json.dumps(solver_options)),
    ],
    workflows_service=argo_svc,
)

created = sim_wf.create()
sim_wf_name = created.metadata.name
print(f"Submitted: {sim_wf_name}")
print(f"Run ID:    {run_id}")

## 4. Watch Real-Time Heatmap

The runner uploads the HDF5 to MinIO every 30s during the solve.
This viewer polls MinIO and shows frames as they appear.

In [ ]:
from tdgl_sdk.viewer import watch_run

# Uses the run_id from the submit cell above.
player = watch_run(store, run_id)
player.display_player()

## 5. Browse Completed Runs

In [ ]:
from tdgl_sdk.viewer import create_player

runs = store.list_runs()
for r in runs:
    rid = r["run_id"][:13]
    frames = r.get("n_frames", "?")
    status = r.get("status", "?")
    created = r.get("created_at", "")[:16]
    print(f"  {rid}  {frames:>6} frames  {status:<10}  {created}")

if runs:
    latest = runs[0]["run_id"]
    print(f"\nTo view: h5 = store.download_h5('{latest[:13]}...'); p = create_player(h5); p.display_player()")